In [ ]:
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

models = [
    '../helmBenchmark/openai_gpt-4o-2024-08-06.pkl',
    '../helmBenchmark/anthropic_claude-3-5-sonnet-20241022.pkl',
    '../helmBenchmark/google_gemini-1.5-pro-002.pkl',
    '../helmBenchmark/amazon_nova-pro-v1:0.pkl',
    '../helmBenchmark/meta_llama-3.1-405b-instruct-turbo.pkl',
    '../helmBenchmark/qwen_qwen2.5-72b-instruct-turbo.pkl',
    '../helmBenchmark/mistralai_mixtral-8x22b.pkl',
    '../helmBenchmark/databricks_dbrx-instruct.pkl',
    '../helmBenchmark/meta_llama-3-8b.pkl',
    '../helmBenchmark/microsoft_phi-3-small-8k-instruct.pkl',
]

TASKS = ['narrative_qa', 'natural_qa', 'wmt_14']
SEED = 1
random.seed(SEED)

def load_df(p):
    df = pd.read_pickle(p)
    df = df[df['task'].isin(TASKS)].copy()
    
    df['reference_text'] = df['references'].apply(lambda x: [r.get('output', {}).get('text') for r in x] if isinstance(x, list) and len(x) > 0 else [])
    df['reference_text'] = df['reference_text'].apply(lambda lst: '\n---\n'.join(lst))   
    df['model'] = p.split('/')[-1].replace('.pkl','')
    
    return df

dfs = [load_df(p) for p in models]

In [ ]:
cleaned_dfs = [df.drop_duplicates(subset='instance_id', keep='first').reset_index(drop=True) for df in dfs]
cleaned_dfs = pd.concat(cleaned_dfs)

In [ ]:
narrative_qa_ids = random.sample(cleaned_dfs.loc[cleaned_dfs['task'] == 'narrative_qa']['instance_id'].value_counts().loc[lambda x: x == 10].index.tolist(), 30)
natural_qa_ids = random.sample(cleaned_dfs.loc[cleaned_dfs['task'] == 'natural_qa']['instance_id'].value_counts().loc[lambda x: x == 10].index.tolist(), 30)

natural_qa = cleaned_dfs[cleaned_dfs['instance_id'].isin(natural_qa_ids)]
narrative_qa = cleaned_dfs[cleaned_dfs['instance_id'].isin(narrative_qa_ids)]

In [ ]:
wmt_dfs = [df[df['task'] == 'wmt_14'] for df in dfs]

common_inputs = set(wmt_dfs[0]['reference_text'])
for d in wmt_dfs[1:]:
    common_inputs &= set(d['reference_text'])

In [ ]:
wmt_dfs = [df[df['task'] == 'wmt_14'] for df in dfs]

common_inputs = set(wmt_dfs[0]['reference_text'])
for d in wmt_dfs[1:]:
    common_inputs &= set(d['reference_text'])

common_inputs = random.sample(list(common_inputs), 30)

all_commons = []
for x in tqdm(common_inputs):
    commons = pd.concat([d.loc[d['reference_text'] == x] for d in wmt_dfs], ignore_index=True)
    all_commons.append(commons)

wmt_14 = pd.concat(all_commons)
print(sum([len(c) for c in all_commons]))

In [ ]:
HELMSamples = pd.concat([narrative_qa, natural_qa, wmt_14])
HELMSamples.to_csv('../results/HELMSamples.csv')